# RoadAnomaly — Stage 5 final test evaluation (one shot)

**Run only after** `FINAL_MODEL_SELECTION_FREEZE.json` exists.

This notebook:
1. Verifies Drive checkpoint SHA-256 against freeze metadata.
2. Verifies split lock.
3. Evaluates E0/E1/E2a/E2b **once** on the test split at frozen operational thresholds.
4. Writes results under Drive `experiments/results/phase4/` and metadata `…/phase4/`.

Does **not** retune thresholds. Does **not** retrain.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadAnomaly")
CLONE_DIR = Path("/content/RoadAnomaly")
GITHUB_REPO = "https://github.com/AhmedChatti/RoadAnomaly.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT_SHA = "d7cc9e4ef2b67acafd7431b2987eaa397c006e13"  # or later commit with this notebook
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"
EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
AUTHORIZE_TEST_EVALUATION = True  # set True only for the approved one-shot run


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

os.environ["ROADANOMALY_ARTIFACTS_ROOT"] = str(DRIVE_ROOT)
os.environ["ROADANOMALY_FRAMES_DIR"] = str(DRIVE_ROOT / "dataset" / "original" / "frames")
os.environ["ROADANOMALY_CHECKPOINTS_DIR"] = str(DRIVE_ROOT / "checkpoints")
os.environ["ROADANOMALY_METADATA_DIR"] = str(DRIVE_ROOT / "experiments" / "metadata")
os.environ["ROADANOMALY_RESULTS_DIR"] = str(DRIVE_ROOT / "experiments" / "results")
os.environ["ROADANOMALY_FIGURES_DIR"] = str(DRIVE_ROOT / "experiments" / "figures")

token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not CLONE_DIR.exists():
    run(["git", "clone", "--branch", GITHUB_BRANCH, auth_url, str(CLONE_DIR)])
else:
    run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
    run(["git", "checkout", GITHUB_BRANCH], cwd=str(CLONE_DIR))
    run(["git", "pull", "origin", GITHUB_BRANCH], cwd=str(CLONE_DIR))
if GITHUB_COMMIT_SHA:
    run(["git", "checkout", GITHUB_COMMIT_SHA], cwd=str(CLONE_DIR))
print("HEAD", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(CLONE_DIR), text=True).strip())
os.environ["ROADANOMALY_ROOT"] = str(CLONE_DIR)
sys.path.insert(0, str(CLONE_DIR / "src"))
os.chdir(CLONE_DIR)

In [ ]:
assert AUTHORIZE_TEST_EVALUATION, "Set AUTHORIZE_TEST_EVALUATION=True only for the frozen one-shot test run."

from dataset import load_split_manifest, verify_split_lock
from evaluation import evaluate_predictor_on_split
from models.baseline import AllBackgroundModel
from losses import bce_dice_loss
import tensorflow as tf
from tensorflow import keras

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_sha = verify_split_lock(load_split_manifest())
assert split_sha == EXPECTED_SPLIT
print("split_ok", split_sha)

freeze_path = CLONE_DIR / "experiments" / "metadata" / "phase4" / "FINAL_MODEL_SELECTION_FREEZE.json"
if not freeze_path.exists():
    freeze_path = DRIVE_ROOT / "experiments" / "metadata" / "phase4" / "FINAL_MODEL_SELECTION_FREEZE.json"
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
print("primary", freeze["primary_model"]["experiment_id"])

out_dir = DRIVE_ROOT / "experiments" / "results" / "phase4"
meta_dir = DRIVE_ROOT / "experiments" / "metadata" / "phase4"
out_dir.mkdir(parents=True, exist_ok=True)
meta_dir.mkdir(parents=True, exist_ok=True)

guard = meta_dir / "STAGE5_TEST_EVAL_DONE.json"
if guard.exists():
    raise RuntimeError(f"Test evaluation already recorded at {guard}. Refusing a second run.")

rows = []
for spec in freeze["models_to_evaluate_once_on_test"]:
    eid = spec["experiment_id"]
    thr = float(spec["operational_threshold"])
    normalize = spec["normalize"]
    print("Evaluating", eid, "thr", thr)

    if eid.startswith("E0"):
        model = AllBackgroundModel()
        predict_fn = lambda x, m=model: m.predict_proba(x)
        ckpt_sha = None
        ckpt_ok = True
    else:
        ckpt = Path(spec["checkpoint_path_drive"])
        assert ckpt.is_file(), ckpt
        ckpt_sha = sha256_file(ckpt)
        expected = spec["checkpoint_sha256"]
        ckpt_ok = ckpt_sha == expected
        print("  sha", ckpt_sha, "match", ckpt_ok)
        assert ckpt_ok, (ckpt_sha, expected)
        loaded = keras.models.load_model(
            ckpt,
            custom_objects={"bce_dice": bce_dice_loss(positive_weight=15.0)},
            compile=False,
        )
        predict_fn = lambda x, m=loaded: m.predict(x, verbose=0)

    result = evaluate_predictor_on_split(
        predict_fn,
        "test",
        threshold=thr,
        normalize=normalize,
        batch_size=4,
        allow_test=True,
    )
    per = result["per_image"]
    per_path = out_dir / f"{eid}_test_per_image.csv"
    per.to_csv(per_path, index=False)
    mp = result["mean_per_image"]
    rows.append({
        "experiment_id": eid,
        "split": "test",
        "threshold": thr,
        "normalize": normalize,
        "checkpoint_sha256": ckpt_sha,
        "checkpoint_sha_match": ckpt_ok,
        **mp,
        "global_dice": result["global"]["dice"],
        "global_iou": result["global"]["iou"],
        "global_accuracy": result["global"]["accuracy"],
        "per_image_csv": str(per_path),
    })

summary = pd.DataFrame(rows)
summary_path = out_dir / "phase4_final_test_comparison.csv"
summary.to_csv(summary_path, index=False)
report = {
    "stage": 5,
    "status": "completed",
    "split": "test",
    "split_manifest_sha256": split_sha,
    "freeze_file": str(freeze_path),
    "primary_model": freeze["primary_model"]["experiment_id"],
    "summary_csv": str(summary_path),
    "rows": rows,
    "notes": "One-shot test evaluation after frozen validation selection. No threshold retuning.",
}
guard.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(summary)
print("Wrote", summary_path)
print("Wrote", guard)